<a href="https://colab.research.google.com/github/himanshu-gangwar/AI_Learning/blob/main/crewai_groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 CrewAI + Groq: Multi-Agent AI Systems
## A Hands-On Tutorial for Learning Agentic AI

---

### 🎯 What You'll Learn

| Concept | Description |
|--------|-------------|
| **Agents** | Autonomous AI workers with roles, goals & backstories |
| **Tasks** | Specific jobs assigned to agents |
| **Tools** | Capabilities agents can use (search, code, etc.) |
| **Crew** | A team of agents working together |
| **Process** | How agents collaborate (sequential / hierarchical) |

### 🏗️ Project: AI Research Report Generator
We'll build a **3-agent crew** that:
1. 🔍 **Researcher Agent** — Gathers information on a topic
2. 🧠 **Analyst Agent** — Analyses and structures findings
3. ✍️ **Writer Agent** — Produces a polished final report

> **Why this project?** It demonstrates the full CrewAI lifecycle clearly — each agent has a distinct role, they pass work to each other, and you see a real output at the end.

---
## 📦 Step 1: Install Dependencies

We need:
- `crewai` — the multi-agent framework
- `crewai-tools` — built-in tools (web search, etc.)
- `groq` — fast LLM inference via Groq's LPU chips

> ⚠️ **Note:** Modern CrewAI (v0.80+) uses LiteLLM under the hood and accepts the LLM as a **string** like `'groq/llama3-70b-8192'` — no need for `langchain-groq` or a `ChatGroq` object anymore!

In [ ]:
# Install all required packages
!pip install crewai crewai-tools groq -q
print("✅ All packages installed!")

# Check crewai version (important — API changed in v0.80+)
import crewai
print(f"   crewai version: {crewai.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.7/781.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## 🔑 Step 2: Configure API Keys

You need a **free Groq API key** from [console.groq.com](https://console.groq.com).

Groq is used instead of OpenAI here because:
- ⚡ Much faster inference (LPU hardware)
- 🆓 Generous free tier
- 🧠 Access to Llama 3, Mixtral, Gemma models

In [ ]:
!pip install litellm -q
print("✅ litellm installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 35.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.13.0 requires pydantic~=2.11.9, but you have pydantic 2.12.5 which is incompatible.
crewai 1.13.0 requires python-dotenv~=1.1.1, but you have python-dotenv 1.0.1 which is incompatible.
c

In [ ]:
import importlib, crewai
importlib.reload(crewai)
from crewai import Agent, Task, Crew, Process
print("✅ crewai reloaded with litellm support")

✅ crewai reloaded with litellm support


In [ ]:
import os
from getpass import getpass

# Securely enter your Groq API key
groq_api_key = getpass("🔑 Enter your Groq API Key: ")

# ─────────────────────────────────────────────
# 🔑 CONCEPT: How CrewAI finds your API key
# ─────────────────────────────────────────────
# Modern CrewAI uses LiteLLM to talk to any LLM provider.
# You just set the env var for your provider — that's it!
# LiteLLM reads GROQ_API_KEY automatically when you use
# the 'groq/...' model string prefix.
# ─────────────────────────────────────────────
os.environ["GROQ_API_KEY"] = groq_api_key

print("✅ API key configured!")
print("   CrewAI will use this via LiteLLM when model='groq/...' is specified.")

🔑 Enter your Groq API Key: ··········
✅ API key configured!
   CrewAI will use this via LiteLLM when model='groq/...' is specified.


---
## 🧠 Step 3: Understanding Agents

### What is an Agent?

An **Agent** is an autonomous AI entity that:
- Has a **role** (who it is)
- Has a **goal** (what it wants to achieve)
- Has a **backstory** (context that shapes its behaviour)
- Can use **tools** to interact with the world
- Is powered by an **LLM** (the brain)

```
┌─────────────────────────────────────┐
│              AGENT                  │
│  ┌─────────┐  ┌──────┐  ┌───────┐  │
│  │  Role   │  │ Goal │  │ Story │  │
│  └─────────┘  └──────┘  └───────┘  │
│         ┌──────────────┐           │
│         │  LLM (Brain) │           │
│         └──────────────┘           │
│    ┌────────┐  ┌────────────────┐  │
│    │ Tools  │  │ Memory/Context │  │
│    └────────┘  └────────────────┘  │
└─────────────────────────────────────┘
```

Let's create our Groq-powered LLM first, then build the agents.

In [ ]:
from crewai import Agent, Task, Crew, Process

# ─────────────────────────────────────────────
# 🧠 CONCEPT: Specifying the LLM in modern CrewAI
# ─────────────────────────────────────────────
# Since CrewAI v0.80+, you pass the LLM as a STRING:
#   'groq/llama3-70b-8192'
#    ^^^^  ^^^^^^^^^^^^^^^
#    |     model name on Groq
#    provider prefix (LiteLLM routing)
#
# CrewAI uses LiteLLM internally, which handles auth
# automatically using the GROQ_API_KEY env var we set.
# No need to create a ChatGroq object!
# ─────────────────────────────────────────────

# The model string we'll use for all agents
GROQ_MODEL = "groq/llama-3.3-70b-versatile"

# Available Groq models (all free tier):
# - groq/llama3-70b-8192   ← best quality, use this
# - groq/llama3-8b-8192    ← faster, smaller
# - groq/mixtral-8x7b-32768 ← large context window
# - groq/gemma2-9b-it      ← Google's Gemma 2

print(f"✅ LLM configured: {GROQ_MODEL}")
print(f"   Provider: Groq (via LiteLLM)")
print(f"   Context window: 8,192 tokens")
print(f"   Auth: automatically reads GROQ_API_KEY env var")

✅ LLM configured: groq/llama-3.3-70b-versatile
   Provider: Groq (via LiteLLM)
   Context window: 8,192 tokens
   Auth: automatically reads GROQ_API_KEY env var


In [ ]:
!pip install litellm -q

# Must restart the crewai module so it picks up litellm
import importlib, sys

# Remove cached crewai modules so they reload with litellm available
mods_to_remove = [key for key in sys.modules if 'crewai' in key]
for mod in mods_to_remove:
    del sys.modules[mod]

# Re-import fresh
from crewai import Agent, Task, Crew, Process

print("✅ litellm installed & crewai reloaded!")
print(f"   Removed & reloaded {len(mods_to_remove)} crewai modules")

✅ litellm installed & crewai reloaded!
   Removed & reloaded 344 crewai modules


In [ ]:
# ─────────────────────────────────────────────
# 🔍 AGENT 1: The Researcher
# ─────────────────────────────────────────────
# Role:      Senior Research Specialist
# Purpose:   Gather comprehensive info on a topic
# Key param: verbose=True lets us see its thinking
# ─────────────────────────────────────────────

researcher = Agent(
    role="Senior Research Specialist",

    goal=(
        "Conduct thorough research on the given topic and compile "
        "comprehensive, accurate information including key concepts, "
        "recent developments, statistics, and expert perspectives."
    ),

    backstory=(
        "You are an expert researcher with 15 years of experience across "
        "technology, science, and business domains. You are known for your "
        "ability to quickly identify the most important aspects of any topic "
        "and present information in a structured, factual manner. "
        "You always cite multiple perspectives and highlight what makes a topic important."
    ),

    # ─────────────────────────────────────────────
    # KEY CHANGE: llm is now a STRING, not an object
    # Format: 'provider/model-name'
    # CrewAI + LiteLLM handle the rest automatically
    # ─────────────────────────────────────────────
    llm=GROQ_MODEL,
    verbose=True,
    allow_delegation=False,
    memory=True
)

print("✅ Agent 1 created: Researcher")
print(f"   Role: {researcher.role}")
print(f"   LLM:  {GROQ_MODEL}")

✅ Agent 1 created: Researcher
   Role: Senior Research Specialist
   LLM:  groq/llama-3.3-70b-versatile


In [ ]:
# ─────────────────────────────────────────────
# 🧠 AGENT 2: The Analyst
# ─────────────────────────────────────────────
# Role:    Strategic Analyst
# Purpose: Takes raw research → finds patterns,
#          insights, implications
# ─────────────────────────────────────────────

analyst = Agent(
    role="Strategic Analyst",

    goal=(
        "Analyse the research findings, identify key patterns and insights, "
        "evaluate implications, and structure the information into clear "
        "sections with actionable takeaways."
    ),

    backstory=(
        "You are a strategic analyst with a background in consulting and data science. "
        "You excel at taking raw information and transforming it into structured insights. "
        "You identify trends, spot contradictions, evaluate risks and opportunities, "
        "and always ask 'so what?' — turning facts into meaningful conclusions. "
        "Your analyses are always logical, well-structured, and actionable."
    ),

    llm=GROQ_MODEL,
    verbose=True,
    allow_delegation=False,
    memory=True
)

print("✅ Agent 2 created: Analyst")
print(f"   Role: {analyst.role}")
print(f"   LLM:  {GROQ_MODEL}")

✅ Agent 2 created: Analyst
   Role: Strategic Analyst
   LLM:  groq/llama-3.3-70b-versatile


In [ ]:
# ─────────────────────────────────────────────
# ✍️ AGENT 3: The Writer
# ─────────────────────────────────────────────
# Role:    Content Writer & Editor
# Purpose: Transforms analysis into polished
#          readable report for the audience
# ─────────────────────────────────────────────

writer = Agent(
    role="Content Writer & Editor",

    goal=(
        "Transform the research and analysis into a compelling, well-structured "
        "report that is engaging, clear, and appropriate for the target audience. "
        "Ensure the writing flows naturally, uses concrete examples, and delivers "
        "real value to the reader."
    ),

    backstory=(
        "You are an experienced technical writer and editor who has written "
        "hundreds of reports, whitepapers, and articles for top publications. "
        "You have a gift for making complex topics accessible without oversimplifying. "
        "Your writing style is clear, engaging, and authoritative. "
        "You always structure content with a compelling introduction, "
        "well-organized body sections, and a memorable conclusion."
    ),

    llm=GROQ_MODEL,
    verbose=True,
    allow_delegation=False,
    memory=True
)

print("✅ Agent 3 created: Writer")
print(f"   Role: {writer.role}")
print(f"   LLM:  {GROQ_MODEL}")
print("\n🎉 All 3 agents ready!")

✅ Agent 3 created: Writer
   Role: Content Writer & Editor
   LLM:  groq/llama-3.3-70b-versatile

🎉 All 3 agents ready!


---
## 📋 Step 4: Understanding Tasks

### What is a Task?

A **Task** is a specific piece of work assigned to an agent. Think of it like a job card:

```
┌──────────────────────────────────────────┐
│                  TASK                    │
│  description  → What exactly to do       │
│  expected_output → What success looks like│
│  agent        → Who does this job        │
│  context      → (optional) prior tasks   │
└──────────────────────────────────────────┘
```

Tasks can be **chained** — Task 2 can receive Task 1's output as context. This is how agents collaborate!

In [ ]:
# ─────────────────────────────────────────────
# 📌 Define the research topic
# ─────────────────────────────────────────────
# Change this to any topic you want!
TOPIC = "Quantum Computing and its impact on Cybersecurity"

print(f"📌 Research Topic: {TOPIC}")
print("\nCreating tasks for each agent...")

📌 Research Topic: Quantum Computing and its impact on Cybersecurity

Creating tasks for each agent...


In [ ]:
# ─────────────────────────────────────────────
# 📋 TASK 1: Research Task → given to Researcher
# ─────────────────────────────────────────────

research_task = Task(
    description=f"""
    Conduct comprehensive research on the following topic:
    **{TOPIC}**

    Your research must cover:
    1. Core concepts and fundamentals
    2. Current state and recent developments (2023-2024)
    3. Key players, companies, or researchers involved
    4. Real-world applications and use cases
    5. Challenges and limitations
    6. Future outlook and predictions

    Compile all findings into a detailed research brief.
    """,

    expected_output="""
    A comprehensive research brief (600-800 words) covering all 6 areas above,
    written in bullet points and short paragraphs, with facts and specifics.
    """,

    agent=researcher  # ← Assigned to the Researcher agent
)

print("✅ Task 1 created: Research Task → Researcher Agent")

✅ Task 1 created: Research Task → Researcher Agent


In [ ]:
# ─────────────────────────────────────────────
# 📋 TASK 2: Analysis Task → given to Analyst
# context=[research_task] means it gets Task 1's output!
# ─────────────────────────────────────────────

analysis_task = Task(
    description=f"""
    Using the research brief provided, perform a strategic analysis of:
    **{TOPIC}**

    Your analysis must include:
    1. **Key Insights**: The 3-5 most important takeaways
    2. **Opportunities**: What doors does this open?
    3. **Risks & Challenges**: What are the critical concerns?
    4. **Stakeholder Impact**: Who is most affected and how?
    5. **Timeline Assessment**: Short-term (1-2yr) vs long-term (5-10yr) outlook
    6. **Recommended Actions**: What should readers/organisations do?
    """,

    expected_output="""
    A structured analysis document with clear headings for each of the 6 sections,
    written in a professional analytical style, 400-600 words.
    """,

    agent=analyst,              # ← Assigned to Analyst
    context=[research_task]     # ← Gets Researcher's output as input!
)

print("✅ Task 2 created: Analysis Task → Analyst Agent")
print("   📨 Will receive output from Task 1 as context")

✅ Task 2 created: Analysis Task → Analyst Agent
   📨 Will receive output from Task 1 as context


In [ ]:
# ─────────────────────────────────────────────
# 📋 TASK 3: Writing Task → given to Writer
# context=[research_task, analysis_task] = both prior outputs!
# ─────────────────────────────────────────────

writing_task = Task(
    description=f"""
    Using both the research brief and the strategic analysis, write a comprehensive
    and engaging report on:
    **{TOPIC}**

    The report must:
    - Begin with a compelling executive summary (100 words)
    - Have clearly structured sections with descriptive headings
    - Explain technical concepts in plain language with examples
    - Include the key insights and recommendations from the analysis
    - End with a forward-looking conclusion
    - Be written for a professional but non-specialist audience
    - Use markdown formatting (##, **, bullet points, etc.)
    """,

    expected_output="""
    A polished, publication-ready report in Markdown format, 800-1000 words,
    with a title, executive summary, 4-5 body sections, and a conclusion.
    """,

    agent=writer,
    context=[research_task, analysis_task]  # ← Gets BOTH prior outputs!
)

print("✅ Task 3 created: Writing Task → Writer Agent")
print("   📨 Will receive outputs from Task 1 AND Task 2 as context")
print("\n📋 All 3 tasks defined!")

✅ Task 3 created: Writing Task → Writer Agent
   📨 Will receive outputs from Task 1 AND Task 2 as context

📋 All 3 tasks defined!


---
## 👥 Step 5: Understanding the Crew

### What is a Crew?

A **Crew** is the team — it orchestrates agents and tasks.

```
                    CREW
        ┌─────────────────────────┐
        │  Process: Sequential    │
        │                         │
        │  Task 1 → Task 2 → Task 3│
        │    ↓         ↓       ↓   │
        │ Research  Analyse  Write  │
        └─────────────────────────┘
```

### Process Types:
| Process | How it works |
|---------|-------------|
| **Sequential** | Tasks run one after another (linear pipeline) |
| **Hierarchical** | A manager agent delegates tasks dynamically |

We'll use **Sequential** — perfect for our research → analyse → write pipeline.

In [ ]:
# ─────────────────────────────────────────────
# 👥 Assemble the CREW
# ─────────────────────────────────────────────

crew = Crew(
    agents=[researcher, analyst, writer],   # All 3 agents
    tasks=[research_task, analysis_task, writing_task],  # All 3 tasks

    process=Process.sequential,  # Run tasks in order
    verbose=True,                # Show detailed logs

    # Optional: Set memory for the whole crew
    memory=False,  # Keep False for now to avoid extra setup
)

print("✅ Crew assembled!")
print(f"   Agents: {len(crew.agents)}")
print(f"   Tasks:  {len(crew.tasks)}")
print(f"   Process: Sequential")
print("\n" + "="*50)
print("🚀 Ready to kickoff! Run the next cell...")
print("="*50)

✅ Crew assembled!
   Agents: 3
   Tasks:  3
   Process: Sequential

🚀 Ready to kickoff! Run the next cell...


---
## 🚀 Step 6: Kickoff the Crew!

This is where the magic happens. When you call `crew.kickoff()`, CrewAI:

1. Sends **Task 1** to the Researcher agent
2. The Researcher thinks, uses the LLM, produces output
3. That output becomes **context** for Task 2 (Analyst)
4. The Analyst analyses and produces output
5. Both outputs become **context** for Task 3 (Writer)
6. The Writer produces the final polished report

> ⏱️ This may take 1-2 minutes depending on Groq's speed. Watch the verbose output to see each agent's reasoning!

In [ ]:
import time

print(f"🚀 Starting Crew for topic: '{TOPIC}'")
print("="*60)
print("Watch as each agent completes their task...\n")

start_time = time.time()

# ─────────────────────────────────────────────
# 🔥 THE MAIN EVENT: kickoff() starts the crew
# ─────────────────────────────────────────────
result = crew.kickoff()

end_time = time.time()
duration = round(end_time - start_time, 1)

print("\n" + "="*60)
print(f"✅ Crew completed in {duration} seconds!")
print("="*60)

🚀 Starting Crew for topic: 'Quantum Computing and its impact on Cybersecurity'
Watch as each agent completes their task...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2ac7fc8b-76d7-4d65-8484-358f360aeda5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Conduct comprehensive research on the following topic:                                                     │
│      **Quantum Computing and its impact on Cybersecurity**                                                      │
│                                                                                                                 │
│      Your research must cover:                                                                                  │
│      1. Core concepts and fundamentals                                                                          │
│      2. Current state and recent developments (2023-2024)                                                       │
│      3. Key players, companies, or researchers involved                                                         │
│      4. Real-world applications and use cases                                                                   │
│      5. Challenges and limitations                                                                              │
│      6. Future outlook and predictions                                                                          │
│                                                                                                                 │
│      Compile all findings into a detailed research brief.                                                       │
│                                                                                                                 │
│  ID: 9ed09b97-a059-4768-ab00-108ef7314a21                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Specialist                                                                              │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Conduct comprehensive research on the following topic:                                                     │
│      **Quantum Computing and its impact on Cybersecurity**                                                      │
│                                                                                                                 │
│      Your research must cover:                                                                                  │
│      1. Core concepts and fundamentals                                                                          │
│      2. Current state and recent developments (2023-2024)                                                       │
│      3. Key players, companies, or researchers involved                                                         │
│      4. Real-world applications and use cases                                                                   │
│      5. Challenges and limitations                                                                              │
│      6. Future outlook and predictions                                                                          │
│                                                                                                                 │
│      Compile all findings into a detailed research brief.                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Specialist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  * Introduction to Quantum Computing and its impact on Cybersecurity:                                           │
│  Quantum computing is a revolutionary technology that uses the principles of quantum mechanics to perform       │
│  calculations and operations on data. This technology has the potential to significantly impact various         │
│  fields, including cybersecurity. Quantum computing can break certain classical encryption algorithms, but it   │
│  can also be used to create new, quantum-resistant encryption methods. The impact of quantum computing on       │
│  cybersecurity is a rapidly evolving field, with new developments and advancements being made regularly.        │
│                                                                                                                 │
│  * Core concepts and fundamentals:                                                                              │
│  Quantum computing is based on the principles of quantum mechanics, including superposition, entanglement, and  │
│  wave function collapse. These principles allow quantum computers to perform certain calculations much faster   │
│  than classical computers. Key concepts in quantum computing include:                                           │
│    + Qubits: the basic units of quantum information                                                             │
│    + Quantum gates: the basic operations used to manipulate qubits                                              │
│    + Quantum algorithms: programs that run on quantum computers                                                 │
│    + Quantum cryptography: the use of quantum mechanics to create secure communication channels                 │
│    + Post-quantum cryptography: the development of new cryptographic algorithms that are resistant to quantum   │
│  computer attacks                                                                                               │
│                                                                                                                 │
│  * Current state and recent developments (2023-2024):                                                           │
│  The current state of quantum computing is one of rapid advancement and development. Several companies,         │
│  including Google, IBM, and Microsoft, are actively working on developing quantum computing technology. Recent  │
│  developments include:                                                                                          │
│    + The development of more powerful quantum computers, such as Google's 53-qubit quantum computer             │
│    + The creation of new quantum algorithms, such as the Quantum Approximate Optimization Algorithm (QAOA)      │
│    + The development of quantum-resistant cryptographic algorithms, such as lattice-based cryptography and      │
│  code-based cryptography                                                                                        │
│    + The establishment of new research centers and initiatives, such as the National Quantum Information        │
│  Science Research Act in the United States                                                                      │
│                                                                                                                 │
│  * Key players, companies, or researchers involved:    

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer & Editor                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Using both the research brief and the strategic analysis, write a comprehensive                            │
│      and engaging report on:                                                                                    │
│      **Quantum Computing and its impact on Cybersecurity**                                                      │
│                                                                                                                 │
│      The report must:                                                                                           │
│      - Begin with a compelling executive summary (100 words)                                                    │
│      - Have clearly structured sections with descriptive headings                                               │
│      - Explain technical concepts in plain language with examples                                               │
│      - Include the key insights and recommendations from the analysis                                           │
│      - End with a forward-looking conclusion                                                                    │
│      - Be written for a professional but non-specialist audience                                                │
│      - Use markdown formatting (##, **, bullet points, etc.)                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer & Editor                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Quantum Computing and its Impact on Cybersecurity                                                           │
│  ### Executive Summary                                                                                          │
│  Quantum computing is a revolutionary technology that uses the principles of quantum mechanics to perform       │
│  calculations and operations on data. This technology has the potential to significantly impact various         │
│  fields, including cybersecurity. Quantum computing can break certain classical encryption algorithms, but it   │
│  can also be used to create new, quantum-resistant encryption methods. As the technology continues to evolve,   │
│  it is essential to stay informed about the latest developments and advancements in the field.                  │
│                                                                                                                 │
│  ## Introduction to Quantum Computing                                                                           │
│  Quantum computing is based on the principles of quantum mechanics, including superposition, entanglement, and  │
│  wave function collapse. These principles allow quantum computers to perform certain calculations much faster   │
│  than classical computers. Key concepts in quantum computing include:                                           │
│  * **Qubits**: the basic units of quantum information                                                           │
│  * **Quantum gates**: the basic operations used to manipulate qubits                                            │
│  * **Quantum algorithms**: programs that run on quantum computers                                               │
│  * **Quantum cryptography**: the use of quantum mechanics to create secure communication channels               │
│  * **Post-quantum cryptography**: the development of new cryptographic algorithms that are resistant to         │
│  quantum computer attacks                                                                                       │
│                                                                                                                 │
│  ## Current State of Quantum Computing                                                                          │
│  The current state of quantum computing is one of rapid advancement and development. Several companies,         │
│  including Google, IBM, and Microsoft, are actively working on developing quantum computing technology. Recent  │
│  developments include:                                                                                          │
│  * The development of more powerful quantum computers, such as Google's 53-qubit quantum computer               │
│  * The creation of new quantum algorithms, such as the Quantum Approximate Optimization Algorithm (QAOA)        │
│  * The development of quantum-resistant cryptographic algorithms, such as lattice-based cryptography and        │
│  code-based cryptography                                                                                        │
│  * The establishment of new research centers and initiatives, such as the National Quantum Information Science  │
│  Research Act in the United States                                                                              │
│                                                        


[2026-04-07 15:26:59][ERROR]: Failed to save to memory: Memory requires an LLM for analysis but initialization failed: Error importing native provider: 1 validation error for OpenAICompletion
  Value error, OPENAI_API_KEY is required [type=value_error, input_value={'model': 'gpt-4o-mini', ...'additional_params': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

To fix this, do one of the following:
  - Set OPENAI_API_KEY for the default model (gpt-4o-mini)
  - Pass a different model: Memory(llm="anthropic/claude-3-haiku-20240307")
  - Pass any LLM instance: Memory(llm=LLM(model="your-model"))
  - To skip LLM analysis, pass all fields explicitly to remember()
    and use depth="shallow" for recall.

Docs: https://docs.crewai.com/concepts/memory


╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Using both the research brief and the strategic analysis, write a comprehensive                            │
│      and engaging report on:                                                                                    │
│      **Quantum Computing and its impact on Cybersecurity**                                                      │
│                                                                                                                 │
│      The report must:                                                                                           │
│      - Begin with a compelling executive summary (100 words)                                                    │
│      - Have clearly structured sections with descriptive headings                                               │
│      - Explain technical concepts in plain language with examples                                               │
│      - Include the key insights and recommendations from the analysis                                           │
│      - End with a forward-looking conclusion                                                                    │
│      - Be written for a professional but non-specialist audience                                                │
│      - Use markdown formatting (##, **, bullet points, etc.)                                                    │
│                                                                                                                 │
│  Agent: Content Writer & Editor                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ Crew completed in 12.1 seconds!


In [ ]:
# ─────────────────────────────────────────────
# 📄 Display the final report
# ─────────────────────────────────────────────
from IPython.display import Markdown, display

print("📄 FINAL REPORT (rendered as Markdown):")
print("="*60)

# Display the result as rendered Markdown
display(Markdown(str(result)))

📄 FINAL REPORT (rendered as Markdown):


## Quantum Computing and its Impact on Cybersecurity
### Executive Summary
Quantum computing is a revolutionary technology that uses the principles of quantum mechanics to perform calculations and operations on data. This technology has the potential to significantly impact various fields, including cybersecurity. Quantum computing can break certain classical encryption algorithms, but it can also be used to create new, quantum-resistant encryption methods. As the technology continues to evolve, it is essential to stay informed about the latest developments and advancements in the field.

## Introduction to Quantum Computing
Quantum computing is based on the principles of quantum mechanics, including superposition, entanglement, and wave function collapse. These principles allow quantum computers to perform certain calculations much faster than classical computers. Key concepts in quantum computing include:
* **Qubits**: the basic units of quantum information
* **Quantum gates**: the basic operations used to manipulate qubits
* **Quantum algorithms**: programs that run on quantum computers
* **Quantum cryptography**: the use of quantum mechanics to create secure communication channels
* **Post-quantum cryptography**: the development of new cryptographic algorithms that are resistant to quantum computer attacks

## Current State of Quantum Computing
The current state of quantum computing is one of rapid advancement and development. Several companies, including Google, IBM, and Microsoft, are actively working on developing quantum computing technology. Recent developments include:
* The development of more powerful quantum computers, such as Google's 53-qubit quantum computer
* The creation of new quantum algorithms, such as the Quantum Approximate Optimization Algorithm (QAOA)
* The development of quantum-resistant cryptographic algorithms, such as lattice-based cryptography and code-based cryptography
* The establishment of new research centers and initiatives, such as the National Quantum Information Science Research Act in the United States

## Key Players and Researchers
Several key players are involved in the development of quantum computing and its impact on cybersecurity. These include:
* **Google**: a leader in the development of quantum computing technology
* **IBM**: a major player in the development of quantum computing hardware and software
* **Microsoft**: a company that is actively developing quantum computing technology and applications
* **Researchers such as Peter Shor**, who developed the first quantum algorithm for factorization, and **Gilles Brassard**, who developed the first quantum cryptography protocol
* **Companies such as Rigetti Computing, IonQ, and D-Wave Systems**, which are developing quantum computing hardware and software

## Real-World Applications and Use Cases
Quantum computing has several real-world applications and use cases, including:
* **Cryptography**: quantum computing can be used to break certain classical encryption algorithms, but it can also be used to create new, quantum-resistant encryption methods
* **Optimization**: quantum computing can be used to solve complex optimization problems, such as those found in logistics and finance
* **Simulation**: quantum computing can be used to simulate complex systems, such as those found in chemistry and materials science
* **Machine learning**: quantum computing can be used to speed up certain machine learning algorithms, such as k-means and support vector machines

## Challenges and Limitations
Despite the potential of quantum computing, there are several challenges and limitations to its development and use. These include:
* **Error correction**: quantum computers are prone to errors due to the fragile nature of quantum states
* **Scalability**: currently, quantum computers are small-scale and need to be scaled up to perform complex calculations
* **Noise**: quantum computers are susceptible to noise, which can cause errors in calculations
* **Standards**: there is a need for standards in quantum computing, particularly in the area of cryptography

## Conclusion
Quantum computing has the potential to significantly impact cybersecurity, both positively and negatively. While it can be used to break certain classical encryption algorithms, it can also be used to create new, quantum-resistant encryption methods. As the technology continues to evolve, it is essential to stay informed about the latest developments and advancements in the field. By understanding the core concepts, current state, and future outlook of quantum computing, individuals and organizations can better prepare for the potential impacts on cybersecurity. The future outlook for quantum computing and its impact on cybersecurity is promising, with widespread adoption of quantum computing technology in various industries, development of new, quantum-resistant cryptographic algorithms and protocols, and establishment of new regulations and standards for the use of quantum computing technology. 

According to a report by **McKinsey**, the quantum computing market is expected to reach $1 billion by 2025. A survey by **IBM** found that 60% of companies are planning to invest in quantum computing technology in the next 5 years. The **National Institute of Standards and Technology (NIST)** has estimated that the development of quantum-resistant cryptographic algorithms could take up to 10 years.

Expert perspectives on the impact of quantum computing on cybersecurity include:
* **Dr. Tal Rabin**, a researcher at IBM, who notes that "quantum computing has the potential to revolutionize the field of cryptography, but it also poses significant challenges."
* **Dr. Scott Aaronson**, a researcher at the University of Texas, who states that "quantum computing is not just a threat to classical cryptography, but also an opportunity to create new, quantum-resistant cryptographic algorithms."
* **Dr. Michele Mosca**, a researcher at the University of Waterloo, who notes that "the development of quantum computing technology is a complex task that requires significant investment and collaboration between industry, academia, and government."

In conclusion, the impact of quantum computing on cybersecurity is a complex and multifaceted topic that requires ongoing research and development. As the technology continues to evolve, it is essential to stay informed about the latest developments and advancements in the field. By understanding the core concepts, current state, and future outlook of quantum computing, individuals and organizations can better prepare for the potential impacts on cybersecurity.

In [ ]:
# Save the report to a file
output_filename = TOPIC.replace(" ", "_").replace("/", "-") + "_report.md"

with open(output_filename, "w") as f:
    f.write(f"# Report: {TOPIC}\n\n")
    f.write(str(result))

print(f"✅ Report saved to: {output_filename}")

✅ Report saved to: Quantum_Computing_and_its_impact_on_Cybersecurity_report.md


---
## 🧪 Step 7: Experiments & Exercises

Now that you understand the basics, try these experiments to deepen your understanding:

### Experiment 1: Change the Topic
Go back to the cell where `TOPIC` is defined and try:
- `"Large Language Models and their business applications"`
- `"Climate change mitigation technologies"`
- `"The future of remote work"`

### Experiment 2: Add a 4th Agent
Try adding a **Fact-Checker** agent between the Analyst and Writer:

```python
fact_checker = Agent(
    role="Fact Checker",
    goal="Verify all claims and flag any inaccuracies",
    backstory="You are a meticulous fact-checker who questions every statistic and claim...",
    llm=llm,
    verbose=True
)
```

### Experiment 3: Give Agents Tools
CrewAI has built-in tools. Try adding web search:

In [ ]:
# ─────────────────────────────────────────────
# BONUS: Example of using CrewAI Tools
# Uncomment to try (requires SERPER_API_KEY for web search)
# ─────────────────────────────────────────────

# from crewai_tools import SerperDevTool, WikipediaQueryRun
# from langchain_community.tools import WikipediaQueryRun
# from langchain_community.utilities import WikipediaAPIWrapper

# # Wikipedia tool (free, no API key needed!)
# wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# # Add the tool to the researcher
# researcher_with_tools = Agent(
#     role="Senior Research Specialist",
#     goal="Conduct thorough research using available tools",
#     backstory="Expert researcher with access to knowledge bases...",
#     llm=llm,
#     tools=[wiki_tool],   # ← Agent can now search Wikipedia!
#     verbose=True
# )

print("💡 Tip: Uncomment the code above to give your researcher Wikipedia access!")
print("   Tools give agents the ability to interact with the real world.")
print("   Available tools: WebSearch, FileRead, CodeExecution, WikipediaSearch, and more!")

💡 Tip: Uncomment the code above to give your researcher Wikipedia access!
   Tools give agents the ability to interact with the real world.
   Available tools: WebSearch, FileRead, CodeExecution, WikipediaSearch, and more!


---
## 🧩 Step 8: Core Concepts Summary

Here's everything you learned in this notebook:

### 🏗️ CrewAI Architecture

```
┌─────────────────────────────────────────────────────────┐
│                        CREW                             │
│  ┌─────────────┐   ┌─────────────┐   ┌─────────────┐  │
│  │   AGENT 1   │   │   AGENT 2   │   │   AGENT 3   │  │
│  │  Researcher │   │   Analyst   │   │   Writer    │  │
│  │  ┌───────┐  │   │  ┌───────┐  │   │  ┌───────┐  │  │
│  │  │ Task1 │──┼──►│  │ Task2 │──┼──►│  │ Task3 │  │  │
│  │  └───────┘  │   │  └───────┘  │   │  └───────┘  │  │
│  └─────────────┘   └─────────────┘   └─────────────┘  │
│                          │                              │
│                    Sequential Flow                      │
│                  Output → Context → Next Task           │
└─────────────────────────────────────────────────────────┘
```

### 📚 Key Concepts

| Concept | Key Parameter | What it does |
|---------|--------------|-------------|
| Agent | `role`, `goal`, `backstory` | Defines who the agent IS |
| Agent | `llm` | The brain (Groq/OpenAI/etc.) |
| Agent | `tools` | What the agent can DO |
| Task | `description` | What to do |
| Task | `expected_output` | What success looks like |
| Task | `context` | Previous task outputs to use |
| Crew | `process` | Sequential vs Hierarchical |
| Crew | `kickoff()` | Starts the whole pipeline |

### 🚀 What to Build Next
- **Customer Support Crew**: Triage → Lookup → Respond → Escalate
- **Code Review Crew**: Write → Review → Test → Document
- **Marketing Crew**: Research → Strategy → Copy → Design brief
- **Data Analysis Crew**: Collect → Clean → Analyse → Visualise

---
> **Resources:**
> - [CrewAI Docs](https://docs.crewai.com)
> - [Groq Console](https://console.groq.com)
> - [CrewAI GitHub](https://github.com/crewAIInc/crewAI)
> - [LangChain Groq](https://python.langchain.com/docs/integrations/chat/groq)

In [ ]:
# ─────────────────────────────────────────────
# 🎓 QUICK QUIZ: Test your understanding!
# ─────────────────────────────────────────────

quiz = {
    "Q1": {
        "question": "What 3 things define an Agent's personality in CrewAI?",
        "answer": "role, goal, and backstory"
    },
    "Q2": {
        "question": "How does Task 2 receive Task 1's output?",
        "answer": "Via the context=[task1] parameter in Task definition"
    },
    "Q3": {
        "question": "What are the 2 process types in CrewAI?",
        "answer": "Process.sequential and Process.hierarchical"
    },
    "Q4": {
        "question": "What method starts the crew execution?",
        "answer": "crew.kickoff()"
    },
    "Q5": {
        "question": "Why are we using Groq instead of OpenAI?",
        "answer": "Groq uses LPU hardware for much faster inference + has a free tier"
    }
}

print("🎓 QUICK QUIZ — Cover the answers and test yourself!")
print("="*60)
for q_id, q_data in quiz.items():
    print(f"\n{q_id}: {q_data['question']}")
    print(f"   💡 Answer: {q_data['answer']}")

🎓 QUICK QUIZ — Cover the answers and test yourself!

Q1: What 3 things define an Agent's personality in CrewAI?
   💡 Answer: role, goal, and backstory

Q2: How does Task 2 receive Task 1's output?
   💡 Answer: Via the context=[task1] parameter in Task definition

Q3: What are the 2 process types in CrewAI?
   💡 Answer: Process.sequential and Process.hierarchical

Q4: What method starts the crew execution?
   💡 Answer: crew.kickoff()

Q5: Why are we using Groq instead of OpenAI?
   💡 Answer: Groq uses LPU hardware for much faster inference + has a free tier
